# Physics 5CL Lab 3, Experiments 4 and 5

This notebook reproduces the numerical report and the exported measurements. The user recalled measuring bright maxima and confirmed plates 222 and 224. Original labels and incomplete entries are retained in the data exports. Additional interpretation choices are documented in `measurement_review.md`.

Run this notebook with its own folder as the working directory. Python and NumPy are required. Numerical values stay unrounded until display. The saved outputs below come from executing the code cells.


In [1]:
from pathlib import Path
import math
import numpy as np
from analyze_lab3 import run_analysis, maximum_coordinates, weighted_scale

root = Path.cwd()
assert (root / 'analyze_lab3.py').exists(), 'Use the notebook folder as the working directory.'
results = run_analysis(root)
print('Regenerated analysis_results.json and both measurement CSV files.')


Regenerated analysis_results.json and both measurement CSV files.


## Wire maxima and fit

For an equivalent slit or wire, the side maxima satisfy $\tan\beta_n=\beta_n$. Write $z_n=\beta_n/\pi$. With opposite-side spans, $D_n=Kz_n$ and $K=2\lambda L/w$.

For independent quoted reading scales $u_n$, the through-origin weighted fit is

$$\begin{aligned}
K&=\frac{\sum_n z_nD_n/u_n^2}{\sum_n z_n^2/u_n^2}, \\
u_K&=\left(\sum_n z_n^2/u_n^2\right)^{-1/2}.
\end{aligned}$$

The wavelength and bench geometry are held fixed. This covariance measures reading uncertainty under the fit model, rather than the larger total uncertainty from feature identification or illumination geometry.


In [2]:
wire = results['wire']
z = maximum_coordinates()
print('Side maximum coordinates beta/pi:', z)
print(f"Fitted K = {wire['K_cm']:.12g} cm")
print(f"Quoted reading scale in K = {wire['K_reading_scale_cm']:.12g} cm")
print(f"Calculated thin diameter = {wire['thin_diameter_um']:.12g} micrometers")
print(f"Diameter reading scale = {wire['thin_diameter_reading_scale_um']:.12g} micrometers")
print('Report display: 116 +/- 2 micrometers')


Side maximum coordinates beta/pi: [1.43029665 2.45902403 3.47088972 4.47740858]
Fitted K = 0.832108825358 cm
Quoted reading scale in K = 0.0126574069815 cm
Calculated thin diameter = 116.33320192 micrometers
Diameter reading scale = 1.76957224499 micrometers
Report display: 116 +/- 2 micrometers


## Fit diagnostic and residual error bars

The normalized residual is $r_n=(D_n-Kz_n)/u_n$. The diagnostic is $\chi^2=\sum_n r_n^2$, with $3-1=2$ degrees of freedom. The input errors are estimated feature-width scales, so the result is interpreted as a mismatch diagnostic rather than a formal probability test.

The residual bars show $u_n/u_n=1$ with the fitted curve held fixed. The horizontal coordinates are theoretical roots and do not carry measured uncertainty.


In [3]:
print('Order   Recorded span   Fitted span   Reading scale   Normalized residual')
for n, measured, predicted, scale, residual in zip(
        range(1, 4), wire['recorded_thin_spans_cm'], wire['fitted_spans_cm'],
        wire['reading_scales_cm'], wire['normalized_residuals']):
    print(f'{n:5d} {measured:15.6f} {predicted:13.6f} {scale:15.6f} {residual:21.6f}')
print(f"Weighted diagnostic = {wire['weighted_chi_squared_diagnostic']:.12g}")
print(f"Reduced weighted diagnostic = {wire['reduced_weighted_diagnostic']:.12g}")
print('Residual plot bars:', wire['residual_plot_error_bars'])


Order   Recorded span   Fitted span   Reading scale   Normalized residual
    1        1.020000      1.190162        0.050000             -3.403249
    2        1.980000      2.046176        0.100000             -0.661756
    3        2.970000      2.888158        0.050000              1.636841
Weighted diagnostic = 14.6992742503
Reduced weighted diagnostic = 7.34963712513
Residual plot bars: [1.0, 1.0, 1.0]


![Wire setup](figures/experiment4_setup.png)

![Wire fit and normalized residuals](figures/experiment4_fit.png)

## Equivalent slit prediction

The equivalent slit has width $a=w$. Its first dark minima are at $y=\pm\lambda L/w=\pm K/2$, giving a central maximum width $K$. Later adjacent minima on one side are separated by $K/2$. The thick wire calculation conditionally interprets the recorded 4-4 span as opposite fourth maxima.


In [4]:
thick = wire['thick']
print(f"Thin slit central width = {wire['K_cm']:.12g} cm")
print(f"Thin adjacent minima spacing = {wire['K_cm']/2:.12g} cm")
print(f"Reading scale in spacing = {wire['K_reading_scale_cm']/2:.12g} cm")
print(f"Thick diameter = {thick['diameter_um']:.12g} micrometers")
print(f"Thick diameter reading scale = {thick['diameter_reading_scale_um']:.12g} micrometers")
print(f"Thick slit central width = {thick['K_cm']:.12g} cm")
print(f"Thick central width reading scale = {thick['K_reading_scale_cm']:.12g} cm")
print('Report central widths: 0.83 +/- 0.01 cm and 0.26 +/- 0.09 cm')


Thin slit central width = 0.832108825358 cm
Thin adjacent minima spacing = 0.416054412679 cm
Reading scale in spacing = 0.00632870349073 cm
Thick diameter = 376.888335925 micrometers
Thick diameter reading scale = 131.091595104 micrometers
Thick slit central width = 0.2568449983 cm
Thick central width reading scale = 0.089337390713 cm
Report central widths: 0.83 +/- 0.01 cm and 0.26 +/- 0.09 cm


## Double slit geometry and missing orders

The label is slit count, slit width, and opaque gap. With center separation $d=a+\mathrm{gap}$, the ratios $d/a$ are 2 for 222 and 3 for 224.

$$\frac{I(\theta)}{I(0)}=\left(\frac{\sin\beta}{\beta}\right)^2\cos^2\alpha$$

Here $\beta=\pi a\sin\theta/\lambda$ and $\alpha=\pi d\sin\theta/\lambda$. A bright interference order disappears when $m=p(d/a)$, where $p$ is a nonzero envelope minimum order. The central order remains bright. Full-profile peaks shift from the pure interference-factor peak positions.


In [5]:
double = results['double_slit']
for pattern, ratio in double['center_separation_to_width_ratio'].items():
    missing = [ratio*p for p in range(1, 4)]
    interior = list(range(-ratio+1, ratio))
    print(f'{pattern}: d/a = {ratio}, missing bright orders {missing}, central-envelope orders {interior}')
print(f"Expected fine-spacing ratio 224/222 = {double['common_geometry_expected_fine_spacing_ratio_224_to_222']:.12g}")


222: d/a = 2, missing bright orders [2, 4, 6], central-envelope orders [-1, 0, 1]
224: d/a = 3, missing bright orders [3, 6, 9], central-envelope orders [-2, -1, 0, 1, 2]
Expected fine-spacing ratio 224/222 = 0.666666666667


![Double slit geometry](figures/experiment5_setup.png)

![Calculated double slit profiles](figures/experiment5_patterns.png)

## Recorded spacing comparison

The first input block is interpreted as 224 following the label correction. Its first two fine-fringe entries give an approximate spacing scale. Its third nominal bright interference order should be missing and is excluded.

For a separate conditional check, the first two envelope entries are treated as the first two envelope side maxima. The predicted third feature is compared with the recorded outer entry. The calculation keeps the original 3.12 cm value. The reported 5.7 is a difference in combined quoted uncertainty scales rather than a probability test.


In [6]:
print(f"224 fine scale = {double['224_fine_first_two_K_cm_per_order']:.12g} cm per order")
print(f"Fine-scale reading uncertainty = {double['224_fine_first_two_uK_cm_per_order']:.12g} cm per order")
print(f"Envelope scale = {double['224_envelope_first_two_K_cm']:.12g} cm")
print(f"Predicted third envelope maximum = {double['224_predicted_third_envelope_maximum_cm']:.12g} cm")
print(f"Prediction reading scale = {double['224_predicted_third_envelope_maximum_scale_cm']:.12g} cm")
print(f"Recorded third feature = {double['224_third_envelope_recorded_cm']:.12g} cm")
print(f"Difference in combined quoted scales = {double['224_third_feature_discrepancy_in_combined_reading_scales']:.12g}")


224 fine scale = 0.234 cm per order
Fine-scale reading uncertainty = 0.022360679775 cm per order
Envelope scale = 0.549803574799 cm
Predicted third envelope maximum = 1.90830757818 cm
Prediction reading scale = 0.0677668484451 cm
Recorded third feature = 3.12 cm
Difference in combined quoted scales = 5.7380223085


## Experiment 5 linear fit diagnostics

The fine fringe fit uses x = m and the envelope fit uses the side maximum coordinates z = beta/pi. Both intercepts are fixed at zero. Each fit uses two measurements and one slope, leaving one degree of freedom. Open triangles in the plot are excluded third features. The lower panels show only fitted readings, with normalized residual bars of plus or minus one quoted reading scale.


In [7]:
from make_experiment5_fit import write_figure
for name, fit in double['linear_fits'].items():
    print(name)
    print('  coordinate:', fit['coordinate'])
    print('  fitted spans (cm):', fit['fitted_spans_cm'])
    print('  normalized residuals:', fit['normalized_residuals'])
    print('  slope (cm):', fit['slope_cm'])
    print('  slope reading scale (cm):', fit['slope_reading_scale_cm'])
    print('  reduced weighted diagnostic:', fit['reduced_weighted_diagnostic'])
    print('  degrees of freedom:', fit['degrees_of_freedom'])
figure_source = write_figure(results, root)
print('Regenerated figures/' + figure_source.name + ' from full precision fit results.')


fine_224
  coordinate: [1.0, 2.0]
  fitted spans (cm): [0.23399999999999999, 0.46799999999999997]
  normalized residuals: [0.3200000000000003, -0.15999999999999903]
  slope (cm): 0.23399999999999999
  slope reading scale (cm): 0.022360679774997897
  reduced weighted diagnostic: 0.12799999999999986
  degrees of freedom: 1
envelope_224
  coordinate: [1.4302966531242025, 2.459024032956761]
  fitted spans (cm): [0.786382212910866, 1.351980203836512]
  normalized residuals: [0.1361778708913408, -0.03960407673023969]
  slope (cm): 0.5498035747990938
  slope reading scale (cm): 0.019524345002877034
  reduced weighted diagnostic: 0.0201128954141534
  degrees of freedom: 1
Regenerated figures/experiment5_fits.tex from full precision fit results.


![Experiment 5 fits and residuals](figures/experiment5_fits.png)


## Provenance and reproduction

The original thicker-wire ambiguous values and the incomplete 222 value are preserved in the CSV files. The repeated 222 envelope entries are excluded from fits. The original plate label is retained alongside its corrected interpretation.

All figure sources are editable LaTeX in `figures/`. Run `python build_figures.py` to compile them and the report. The PNGs above are supplied previews of the same figures used in the report. The notebook regenerates numerical results, rather than overwriting the report prose.

Physics sources: [Cornell Slitfilm appendix, page 8](https://web.physics.wustl.edu/introphys/Archives/SP13/DI.pdf) and [Cornell interference notes](https://muchomas.lassp.cornell.edu/p214/Notes/Interference/notes.html).
